In [1]:
import pandas as pd
import duckdb
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

# Project paths
CERT_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cert\r4.2")
INTERIM_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Verify path
print(f"CERT_DIR exists: {CERT_DIR.exists()}")
print(f"\nFiles in CERT_DIR:\n")

# List all files with sizes
all_files = sorted(CERT_DIR.rglob("*"))
print(f"{'Relative Path':<40} {'Size':>15}")
print("-" * 60)
for f in all_files:
    if f.is_file():
        size = f.stat().st_size
        if size > 1e9:
            size_str = f"{size/1e9:.2f} GB"
        elif size > 1e6:
            size_str = f"{size/1e6:.1f} MB"
        elif size > 1e3:
            size_str = f"{size/1e3:.1f} KB"
        else:
            size_str = f"{size} B"
        rel = str(f.relative_to(CERT_DIR))
        print(f"{rel:<40} {size_str:>15}")

CERT_DIR exists: True

Files in CERT_DIR:

Relative Path                                       Size
------------------------------------------------------------
device.csv                                       29.0 MB
email.csv                                        1.36 GB
file.csv                                        193.1 MB
http.csv                                        14.54 GB
LDAP\2009-12.csv                                150.6 KB
LDAP\2010-01.csv                                150.6 KB
LDAP\2010-02.csv                                148.9 KB
LDAP\2010-03.csv                                147.8 KB
LDAP\2010-04.csv                                147.5 KB
LDAP\2010-05.csv                                146.6 KB
LDAP\2010-06.csv                                145.7 KB
LDAP\2010-07.csv                                144.6 KB
LDAP\2010-08.csv                                142.5 KB
LDAP\2010-09.csv                                140.5 KB
LDAP\2010-10.csv                         

In [6]:
import pandas as pd
import duckdb
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Project paths
CERT_PARENT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cert")
CERT_DIR = CERT_PARENT / "r4.2"
INTERIM_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim")

# List parent folder
print(f"Files in CERT_PARENT ({CERT_PARENT}):")
print("-" * 70)
for f in sorted(CERT_PARENT.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<40} {size_kb:>10.1f} KB")
    elif f.is_dir():
        print(f"  {f.name}/  (directory)")

Files in CERT_PARENT (C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cert):
----------------------------------------------------------------------
  answers/  (directory)
  r4.2/  (directory)
  SEI_Insider_README.txt                          1.4 KB


In [7]:
# Connect to a NEW DuckDB file specifically for CERT data
# Keeping CERT separate from CICIDS for now — we'll merge during warehouse build
CERT_DUCKDB_PATH = INTERIM_DIR / "cert_eda.duckdb"
con = duckdb.connect(str(CERT_DUCKDB_PATH))

# Load psychometric scores (1000 rows, 45 KB — instant)
con.execute(f"""
    CREATE OR REPLACE TABLE raw_psychometric AS
    SELECT * FROM read_csv_auto('{(CERT_DIR / "psychometric.csv").as_posix()}')
""")
psych_count = con.execute("SELECT COUNT(*) FROM raw_psychometric").fetchone()[0]
print(f"✅ raw_psychometric: {psych_count:,} rows")

# Quick stats on OCEAN scores
print("\n=== OCEAN score distributions ===")
print(con.execute("""
    SELECT 
        ROUND(AVG("O"), 1) AS avg_O, ROUND(STDDEV("O"), 1) AS std_O,
        ROUND(AVG("C"), 1) AS avg_C, ROUND(STDDEV("C"), 1) AS std_C,
        ROUND(AVG("E"), 1) AS avg_E, ROUND(STDDEV("E"), 1) AS std_E,
        ROUND(AVG("A"), 1) AS avg_A, ROUND(STDDEV("A"), 1) AS std_A,
        ROUND(AVG("N"), 1) AS avg_N, ROUND(STDDEV("N"), 1) AS std_N
    FROM raw_psychometric
""").fetchdf())

✅ raw_psychometric: 1,000 rows

=== OCEAN score distributions ===
   avg_O  std_O  avg_C  std_C  avg_E  std_E  avg_A  std_A  avg_N  std_N
0   33.2   10.6   30.7   11.3   29.2   11.0   28.8   11.2   29.6    4.9


In [8]:
# Load all 18 LDAP snapshots — one combined table with snapshot_date column
ldap_files = sorted((CERT_DIR / "LDAP").glob("*.csv"))
print(f"Loading {len(ldap_files)} LDAP snapshots...")

# Drop existing table
con.execute("DROP TABLE IF EXISTS raw_ldap")

# Load first file with snapshot_date derived
first_file = ldap_files[0]
snapshot_date = first_file.stem  # e.g., '2009-12'
con.execute(f"""
    CREATE TABLE raw_ldap AS
    SELECT *, CAST('{snapshot_date}-01' AS DATE) AS snapshot_date
    FROM read_csv_auto('{first_file.as_posix()}')
""")

# Append the rest
for f in ldap_files[1:]:
    snapshot_date = f.stem
    con.execute(f"""
        INSERT INTO raw_ldap 
        SELECT *, CAST('{snapshot_date}-01' AS DATE) AS snapshot_date
        FROM read_csv_auto('{f.as_posix()}')
    """)

ldap_count = con.execute("SELECT COUNT(*) FROM raw_ldap").fetchone()[0]
unique_users = con.execute("SELECT COUNT(DISTINCT user_id) FROM raw_ldap").fetchone()[0]
print(f"✅ raw_ldap: {ldap_count:,} rows across {len(ldap_files)} snapshots")
print(f"   Unique users seen across all months: {unique_users:,}")

# Population over time (employee count drops as people leave)
print("\n=== Employee count by month ===")
print(con.execute("""
    SELECT snapshot_date, COUNT(*) AS active_employees
    FROM raw_ldap
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""").fetchdf().to_string(index=False))

Loading 18 LDAP snapshots...
✅ raw_ldap: 16,743 rows across 18 snapshots
   Unique users seen across all months: 1,000

=== Employee count by month ===
snapshot_date  active_employees
   2009-12-01              1000
   2010-01-01              1000
   2010-02-01               989
   2010-03-01               982
   2010-04-01               980
   2010-05-01               974
   2010-06-01               968
   2010-07-01               961
   2010-08-01               947
   2010-09-01               934
   2010-10-01               918
   2010-11-01               906
   2010-12-01               890
   2011-01-01               878
   2011-02-01               868
   2011-03-01               857
   2011-04-01               846
   2011-05-01               845


In [9]:
# Sanity check: do all psychometric users exist in LDAP?
print("\n=== Cross-check: psychometric users in LDAP ===")
print(con.execute("""
    SELECT 
        (SELECT COUNT(*) FROM raw_psychometric) AS psychometric_users,
        (SELECT COUNT(DISTINCT user_id) FROM raw_ldap) AS ldap_unique_users,
        (SELECT COUNT(DISTINCT p.user_id) 
         FROM raw_psychometric p 
         JOIN raw_ldap l ON p.user_id = l.user_id) AS users_in_both
""").fetchdf())

# Distribution of roles
print("\n=== Top 10 roles in latest LDAP snapshot (May 2011) ===")
print(con.execute("""
    SELECT role, COUNT(*) AS employee_count
    FROM raw_ldap
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM raw_ldap)
    GROUP BY role
    ORDER BY employee_count DESC
    LIMIT 10
""").fetchdf())


=== Cross-check: psychometric users in LDAP ===
   psychometric_users  ldap_unique_users  users_in_both
0                1000               1000           1000

=== Top 10 roles in latest LDAP snapshot (May 2011) ===
                      role  employee_count
0     ProductionLineWorker             156
1               Technician             130
2                 Salesman             126
3                Scientist              37
4         SoftwareEngineer              36
5  AdministrativeAssistant              32
6        ComputerScientist              26
7       ElectricalEngineer              25
8       MechanicalEngineer              24
9                  Manager              24


In [10]:
ANSWERS_DIR = CERT_PARENT / "answers"

print(f"Contents of {ANSWERS_DIR}:")
print("-" * 70)
for f in sorted(ANSWERS_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        rel = f.relative_to(ANSWERS_DIR)
        print(f"  {str(rel):<60} {size_kb:>10.1f} KB")
    elif f.is_dir() and f != ANSWERS_DIR:
        rel = f.relative_to(ANSWERS_DIR)
        print(f"  {str(rel)}/  (directory)")

Contents of C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cert\answers:
----------------------------------------------------------------------
  insiders.csv                                                       13.5 KB
  license.txt                                                         3.8 KB
  r2.csv                                                              1.9 KB
  r3.1-1.csv                                                          2.6 KB
  r3.1-2.csv                                                         77.2 KB
  r3.2-1.csv                                                          1.8 KB
  r3.2-2.csv                                                         88.0 KB
  r4.1-1.csv                                                          1.9 KB
  r4.1-2.csv                                                         88.9 KB
  r4.1-3.csv                                                          5.5 KB
  r4.2-1/  (directory)
  r4.2-1\r4.2-1-AAM0658.csv                   

In [11]:
# Are there meaningful score distributions?
print("=== OCEAN distribution (5-bin histogram per dimension) ===\n")
for dim in ['O', 'C', 'E', 'A', 'N']:
    bins = con.execute(f"""
        SELECT 
            CASE 
                WHEN "{dim}" < 10 THEN '00-09'
                WHEN "{dim}" < 20 THEN '10-19'
                WHEN "{dim}" < 30 THEN '20-29'
                WHEN "{dim}" < 40 THEN '30-39'
                ELSE '40+'
            END AS bucket,
            COUNT(*) AS cnt
        FROM raw_psychometric
        GROUP BY bucket
        ORDER BY bucket
    """).fetchdf()
    print(f"{dim} ({['Openness','Conscientiousness','Extraversion','Agreeableness','Neuroticism']['OCEAN'.index(dim)]}):")
    for _, r in bins.iterrows():
        print(f"  {r['bucket']:<8} {'█' * (r['cnt'] // 30)} {r['cnt']}")
    print()

=== OCEAN distribution (5-bin histogram per dimension) ===

O (Openness):
  10-19    █████ 156
  20-29    █████ 160
  30-39    ██████████ 318
  40+      ████████████ 366

C (Conscientiousness):
  10-19    ███████ 234
  20-29    ██████ 203
  30-39    █████████ 276
  40+      █████████ 287

E (Extraversion):
  10-19    ████████ 256
  20-29    ████████ 255
  30-39    ████████ 251
  40+      ███████ 238

A (Agreeableness):
  10-19    ████████ 259
  20-29    ████████ 269
  30-39    ███████ 232
  40+      ████████ 240

N (Neuroticism):
  10-19     19
  20-29    ████████████████ 483
  30-39    ███████████████ 477
  40+       21



In [12]:
import pandas as pd

ANSWERS_DIR = CERT_PARENT / "answers"

# Approach 1: Load insiders.csv if it has the user list
print("=== insiders.csv inspection ===")
insiders_df = pd.read_csv(ANSWERS_DIR / "insiders.csv")
print(f"Columns: {insiders_df.columns.tolist()}")
print(f"Rows: {len(insiders_df)}")
print(f"\nFirst 5 rows:")
print(insiders_df.head().to_string())

# Filter for r4.2 entries
print(f"\n=== Distinct dataset values ===")
if 'dataset' in insiders_df.columns:
    print(insiders_df['dataset'].value_counts())
elif 'release' in insiders_df.columns:
    print(insiders_df['release'].value_counts())
else:
    print("Will need to use file directory approach instead")

=== insiders.csv inspection ===
Columns: ['dataset', 'scenario', 'details', 'user', 'start', 'end']
Rows: 191

First 5 rows:
   dataset  scenario     details     user                start                  end
0      2.0         1      r2.csv  ONS0995     3/6/2010 1:41:56    3/20/2010 8:10:12
1      3.1         1  r3.1-1.csv  CSF0929  07/01/2010 01:24:58  07/16/2010 06:52:00
2      3.1         2  r3.1-2.csv  CCH0959  08/02/2010 10:34:31  09/30/2010 15:04:03
3      3.2         1  r3.2-1.csv  RCW0822  09/29/2010 21:10:27  10/15/2010 06:34:52
4      3.2         2  r3.2-2.csv  JCE0258  07/12/2010 08:16:02  09/03/2010 16:16:29

=== Distinct dataset values ===
dataset
5.2    99
4.2    70
6.1     5
6.2     5
5.1     4
4.1     3
3.1     2
3.2     2
2.0     1
Name: count, dtype: int64


In [17]:
# Approach 2: List the per-user files in r4.2-1 directory
r42_answers = ANSWERS_DIR / "r4.2-1"
malicious_files = sorted(r42_answers.glob("*.csv"))

# Extract user IDs from filenames (format: r4.2-1-USER0123.csv)
malicious_user_ids = []
for f in malicious_files:
    # Extract everything between "r4.2-1-" and ".csv"
    user_id = f.stem.replace("r4.2-1-", "")
    malicious_user_ids.append(user_id)

print(f"=== Malicious user IDs from r4.2-1/ folder ===")
print(f"Total: {len(malicious_user_ids)} malicious insiders")
print(f"\nFirst 10 user IDs: {malicious_user_ids[:10]}")
print(f"Last 10 user IDs:  {malicious_user_ids[-10:]}")

=== Malicious user IDs from r4.2-1/ folder ===
Total: 30 malicious insiders

First 10 user IDs: ['AAM0658', 'AJR0932', 'BDV0168', 'BIH0745', 'BLS0678', 'BTL0226', 'CAH0936', 'DCH0843', 'EHB0824', 'EHD0584']
Last 10 user IDs:  ['MAR0955', 'MAS0025', 'MCF0600', 'MYD0978', 'PPF0435', 'RAB0589', 'RGG0064', 'RKD0604', 'TAP0551', 'WDD0366']


In [14]:
# Validate: are these users in our LDAP/psychometric foundation?
malicious_ids_str = "', '".join(malicious_user_ids)

print(f"\n=== Cross-check malicious users against LDAP & psychometric ===")
result = con.execute(f"""
    SELECT 
        '{len(malicious_user_ids)}' AS total_malicious,
        (SELECT COUNT(DISTINCT user_id) FROM raw_ldap WHERE user_id IN ('{malicious_ids_str}')) AS in_ldap,
        (SELECT COUNT(*) FROM raw_psychometric WHERE user_id IN ('{malicious_ids_str}')) AS in_psychometric
""").fetchdf()
print(result)

# Save the malicious user list to a table for later joins
malicious_df = pd.DataFrame({
    'user_id': malicious_user_ids,
    'is_malicious': 1,
    'scenario': 'r4.2-1'  # all 70 belong to r4.2-1 mixture of 3 scenarios
})

con.register('malicious_temp', malicious_df)
con.execute("""
    CREATE OR REPLACE TABLE dim_malicious_users AS
    SELECT user_id, is_malicious, scenario
    FROM malicious_temp
""")
con.unregister('malicious_temp')

print(f"\n✅ dim_malicious_users created with {len(malicious_df)} entries")


=== Cross-check malicious users against LDAP & psychometric ===
  total_malicious  in_ldap  in_psychometric
0              30       30               30

✅ dim_malicious_users created with 30 entries


In [21]:
import pandas as pd

ANSWERS_DIR = CERT_PARENT / "answers"

# Load insiders.csv — the authoritative answer key for ALL CMU releases
insiders_df = pd.read_csv(ANSWERS_DIR / "insiders.csv")

# Filter to r4.2 only
r42_insiders = insiders_df[insiders_df['dataset'] == 4.2].copy()
print(f"Total malicious insiders in r4.2: {len(r42_insiders)}")
print(f"Scenarios: {sorted(r42_insiders['scenario'].unique())}")

# Per-scenario count
print(f"\n=== Malicious users per scenario ===")
print(r42_insiders.groupby('scenario').size())

# Show all 70 with their attack windows
print(f"\n=== All r4.2 malicious users with attack windows ===")
print(r42_insiders[['user', 'scenario', 'start', 'end']].to_string(index=False))

Total malicious insiders in r4.2: 70
Scenarios: [np.int64(1), np.int64(2), np.int64(3)]

=== Malicious users per scenario ===
scenario
1    30
2    30
3    10
dtype: int64

=== All r4.2 malicious users with attack windows ===
   user  scenario               start                 end
AAM0658         1 10/23/2010 01:34:19 10/29/2010 05:23:28
AJR0932         1 09/10/2010 19:12:01 09/18/2010 02:02:51
BDV0168         1 07/30/2010 19:56:44 08/10/2010 05:16:41
BIH0745         1 07/13/2010 20:15:23 07/13/2010 21:20:44
BLS0678         1 09/21/2010 01:16:22 09/30/2010 04:48:19
BTL0226         1 10/06/2010 22:25:52 10/14/2010 06:43:29
CAH0936         1 08/11/2010 04:00:08 08/12/2010 23:56:19
DCH0843         1 02/04/2011 07:08:00 02/04/2011 07:36:05
EHB0824         1 07/22/2010 21:48:43 07/29/2010 01:08:41
EHD0584         1 10/02/2010 03:46:16 10/08/2010 22:26:26
FMG0527         1 01/05/2011 21:53:29 01/12/2011 01:15:35
FTM0406         1 11/25/2010 06:35:11 12/02/2010 00:35:19
GHL0460         1 11

In [22]:
# Validate: are all 70 users present in our LDAP & psychometric foundation?
malicious_user_ids = r42_insiders['user'].tolist()
malicious_ids_str = "', '".join(malicious_user_ids)

print(f"=== Cross-check 70 malicious users vs LDAP & psychometric ===")
result = con.execute(f"""
    SELECT 
        {len(malicious_user_ids)} AS total_malicious,
        (SELECT COUNT(DISTINCT user_id) FROM raw_ldap WHERE user_id IN ('{malicious_ids_str}')) AS in_ldap,
        (SELECT COUNT(*) FROM raw_psychometric WHERE user_id IN ('{malicious_ids_str}')) AS in_psychometric
""").fetchdf()
print(result)

=== Cross-check 70 malicious users vs LDAP & psychometric ===
   total_malicious  in_ldap  in_psychometric
0               70       70               70


In [23]:
# Replace dim_malicious_users with the full 70-user list
# Parse start/end dates so we can use them for activity labeling later
r42_insiders['start_dt'] = pd.to_datetime(r42_insiders['start'], dayfirst=True, errors='coerce')
r42_insiders['end_dt']   = pd.to_datetime(r42_insiders['end'],   dayfirst=True, errors='coerce')

# Build the dim table
malicious_df = pd.DataFrame({
    'user_id': r42_insiders['user'].values,
    'scenario': r42_insiders['scenario'].astype(int).values,
    'attack_start': r42_insiders['start_dt'].values,
    'attack_end': r42_insiders['end_dt'].values,
    'is_malicious': 1
})

# Drop and recreate
con.execute("DROP TABLE IF EXISTS dim_malicious_users")
con.register('malicious_temp', malicious_df)
con.execute("""
    CREATE TABLE dim_malicious_users AS
    SELECT user_id, scenario, attack_start, attack_end, is_malicious
    FROM malicious_temp
""")
con.unregister('malicious_temp')

# Verify
print("✅ dim_malicious_users recreated")
print(con.execute("""
    SELECT scenario, COUNT(*) AS users, MIN(attack_start) AS earliest_attack, MAX(attack_end) AS latest_attack
    FROM dim_malicious_users
    GROUP BY scenario
    ORDER BY scenario
""").fetchdf())

✅ dim_malicious_users recreated
   scenario  users     earliest_attack       latest_attack
0         1     30 2010-07-07 20:05:29 2011-03-03 01:01:02
1         2     30 2010-06-14 09:25:31 2011-04-14 20:58:56
2         3     10 2010-06-10 07:54:10 2011-04-29 20:04:27


In [24]:
# Read scenarios.txt for context on what each scenario means
scenarios_path = ANSWERS_DIR / "scenarios.txt"
print("=== scenarios.txt ===\n")
with open(scenarios_path, 'r', encoding='utf-8') as f:
    print(f.read())

=== scenarios.txt ===

Brief Scenario Descriptions

1. User who did not previously use removable drives or work after
hours begins logging in after hours, using a removable drive, and
uploading data to wikileaks.org. Leaves the organization shortly
thereafter.

2. User begins surfing job websites and soliciting employment from a
competitor. Before leaving the company, they use a thumb drive (at
markedly higher rates than their previous activity) to steal data.

3. System administrator becomes disgruntled. Downloads a keylogger and
uses a thumb drive to transfer it to his supervisor's machine. The
next day, he uses the collected keylogs to log in as his supervisor
and send out an alarming mass email, causing panic in the
organization. He leaves the organization immediately.

4. A user logs into another user's machine and searches for
interesting files, emailing to their home email. This behavior occurs
more and more frequently over a 3 month period.

5. A member of a group decimated by 

In [25]:
import duckdb

# Load logon.csv — full load, no content column to drop
print("Loading logon.csv (58 MB)...")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_logon AS
    SELECT * FROM read_csv_auto(
        '{(CERT_DIR / "logon.csv").as_posix()}',
        sample_size=-1,
        ignore_errors=false
    )
""")

cnt = con.execute("SELECT COUNT(*) FROM raw_logon").fetchone()[0]
print(f"✅ raw_logon: {cnt:,} rows")

# Schema check
print("\nSchema:")
print(con.execute("DESCRIBE raw_logon").fetchdf())

# Activity types
print("\n=== Activity types ===")
print(con.execute("""
    SELECT activity, COUNT(*) AS cnt
    FROM raw_logon
    GROUP BY activity
    ORDER BY cnt DESC
""").fetchdf())

# Sanity check: are dates parseable?
print("\n=== Sample of date strings ===")
print(con.execute("SELECT DISTINCT date FROM raw_logon LIMIT 5").fetchdf())

# Date range
print("\n=== Earliest/latest events (text comparison) ===")
print(con.execute("""
    SELECT MIN(date) AS earliest_text, MAX(date) AS latest_text
    FROM raw_logon
""").fetchdf())

Loading logon.csv (58 MB)...
✅ raw_logon: 854,859 rows

Schema:
  column_name column_type null   key default extra
0          id     VARCHAR  YES  None    None  None
1        date     VARCHAR  YES  None    None  None
2        user     VARCHAR  YES  None    None  None
3          pc     VARCHAR  YES  None    None  None
4    activity     VARCHAR  YES  None    None  None

=== Activity types ===
  activity     cnt
0    Logon  470591
1   Logoff  384268

=== Sample of date strings ===
                  date
0  03/02/2011 08:04:00
1  03/02/2011 08:36:00
2  03/02/2011 08:41:00
3  03/02/2011 08:42:00
4  03/02/2011 08:48:00

=== Earliest/latest events (text comparison) ===
         earliest_text          latest_text
0  01/01/2011 00:15:05  12/31/2010 23:44:05


In [26]:
print("=== Logon volume: malicious vs benign users ===")
print(con.execute("""
    SELECT 
        CASE WHEN m.user_id IS NOT NULL THEN 'Malicious' ELSE 'Benign' END AS user_class,
        COUNT(DISTINCT l.user) AS unique_users,
        COUNT(*) AS logon_events
    FROM raw_logon l
    LEFT JOIN dim_malicious_users m ON l.user = m.user_id
    GROUP BY user_class
""").fetchdf())

# Breakdown of malicious user logons inside vs outside attack windows
# This is the labeled training data for ML
print("\n=== Malicious user logons: inside vs outside attack windows ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN m.user_id IS NULL THEN 'Benign user'
            WHEN STRPTIME(l.date, '%m/%d/%Y %H:%M:%S') BETWEEN m.attack_start AND m.attack_end 
                THEN 'Malicious user (during attack window)'
            ELSE 'Malicious user (outside attack window)'
        END AS event_class,
        COUNT(*) AS logon_events
    FROM raw_logon l
    LEFT JOIN dim_malicious_users m ON l.user = m.user_id
    GROUP BY event_class
    ORDER BY logon_events DESC
""").fetchdf())

=== Logon volume: malicious vs benign users ===
  user_class  unique_users  logon_events
0     Benign           930        805332
1  Malicious            70         49527

=== Malicious user logons: inside vs outside attack windows ===
                              event_class  logon_events
0                             Benign user        805332
1  Malicious user (outside attack window)         46079
2   Malicious user (during attack window)          3448


In [27]:
import duckdb

# Add a parsed timestamp column to raw_logon
con.execute("""
    CREATE OR REPLACE TABLE raw_logon AS
    SELECT 
        id, user, pc, activity,
        date AS date_text,
        TRY_STRPTIME(date, '%m/%d/%Y %H:%M:%S') AS event_time
    FROM raw_logon
""")

# Verify parse rate
print("=== Parse validation ===")
print(con.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(event_time) AS parsed_rows,
        COUNT(*) - COUNT(event_time) AS parse_failed
    FROM raw_logon
""").fetchdf())

# True date range
print("\n=== True chronological range ===")
print(con.execute("""
    SELECT MIN(event_time) AS earliest, MAX(event_time) AS latest
    FROM raw_logon
""").fetchdf())

# Distribution by month
print("\n=== Logon events by month ===")
print(con.execute("""
    SELECT 
        DATE_TRUNC('month', event_time) AS month,
        COUNT(*) AS events
    FROM raw_logon
    GROUP BY month
    ORDER BY month
""").fetchdf())

# Hour-of-day distribution (afterwards we'll see business vs after-hours patterns)
print("\n=== Hour-of-day distribution (top 5) ===")
print(con.execute("""
    SELECT 
        EXTRACT(HOUR FROM event_time) AS hour,
        COUNT(*) AS events
    FROM raw_logon
    GROUP BY hour
    ORDER BY events DESC
    LIMIT 5
""").fetchdf())

=== Parse validation ===
   total_rows  parsed_rows  parse_failed
0      854859       854859             0

=== True chronological range ===
             earliest              latest
0 2010-01-02 06:49:00 2011-05-17 06:43:35

=== Logon events by month ===
        month  events
0  2010-01-01   53857
1  2010-02-01   53093
2  2010-03-01   60645
3  2010-04-01   55110
4  2010-05-01   52552
5  2010-06-01   56714
6  2010-07-01   53832
7  2010-08-01   55362
8  2010-09-01   52168
9  2010-10-01   51682
10 2010-11-01   48389
11 2010-12-01   47207
12 2011-01-01   48321
13 2011-02-01   45709
14 2011-03-01   51521
15 2011-04-01   44403
16 2011-05-01   24294

=== Hour-of-day distribution (top 5) ===
   hour  events
0     7  149697
1     8  141848
2    17   92291
3    18   84498
4    16   82514


In [28]:
print("Loading device.csv (29 MB)...")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_device AS
    SELECT 
        id, user, pc, activity,
        date AS date_text,
        TRY_STRPTIME(date, '%m/%d/%Y %H:%M:%S') AS event_time
    FROM read_csv_auto(
        '{(CERT_DIR / "device.csv").as_posix()}',
        sample_size=-1,
        ignore_errors=false
    )
""")

cnt = con.execute("SELECT COUNT(*) FROM raw_device").fetchone()[0]
parsed = con.execute("SELECT COUNT(event_time) FROM raw_device").fetchone()[0]
print(f"✅ raw_device: {cnt:,} rows, {parsed:,} parsed ({cnt-parsed} failures)")

# Activity types
print("\n=== Activity types ===")
print(con.execute("""
    SELECT activity, COUNT(*) AS cnt
    FROM raw_device
    GROUP BY activity
    ORDER BY cnt DESC
""").fetchdf())

# Date range
print("\n=== Date range ===")
print(con.execute("""
    SELECT MIN(event_time) AS earliest, MAX(event_time) AS latest
    FROM raw_device
""").fetchdf())

# Hour distribution — USB activity should be more spread out than logons
print("\n=== Hour-of-day top 10 ===")
print(con.execute("""
    SELECT EXTRACT(HOUR FROM event_time) AS hour, COUNT(*) AS events
    FROM raw_device
    GROUP BY hour
    ORDER BY events DESC
    LIMIT 10
""").fetchdf())

Loading device.csv (29 MB)...
✅ raw_device: 405,380 rows, 405,380 parsed (0 failures)

=== Activity types ===
     activity     cnt
0     Connect  203339
1  Disconnect  202041

=== Date range ===
             earliest              latest
0 2010-01-02 07:21:06 2011-05-16 23:22:34

=== Hour-of-day top 10 ===
   hour  events
0    14   46288
1    15   45911
2     9   43684
3    10   40718
4    13   40240
5    11   35827
6    12   35346
7    16   32689
8     8   32015
9    17   18616


In [29]:
# USB activity pattern: malicious vs benign
print("=== USB activity volume by user class ===")
print(con.execute("""
    SELECT 
        CASE WHEN m.user_id IS NOT NULL THEN 'Malicious' ELSE 'Benign' END AS user_class,
        COUNT(DISTINCT d.user) AS unique_users,
        COUNT(*) AS device_events,
        ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT d.user), 0) AS avg_events_per_user
    FROM raw_device d
    LEFT JOIN dim_malicious_users m ON d.user = m.user_id
    GROUP BY user_class
""").fetchdf())

# More telling: USB activity DURING attack windows vs OUTSIDE
print("\n=== USB activity: in/out of attack windows ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN m.user_id IS NULL THEN 'Benign user'
            WHEN d.event_time BETWEEN m.attack_start AND m.attack_end 
                THEN 'Malicious user (during attack window)'
            ELSE 'Malicious user (outside attack window)'
        END AS event_class,
        COUNT(*) AS device_events
    FROM raw_device d
    LEFT JOIN dim_malicious_users m ON d.user = m.user_id
    GROUP BY event_class
    ORDER BY device_events DESC
""").fetchdf())

# Hour-of-day distribution: malicious DURING attack vs everyone else
# This is the smoking gun for Scenario 1 (after-hours USB use)
print("\n=== USB hour-of-day: attack-window events vs benign ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN m.user_id IS NOT NULL AND d.event_time BETWEEN m.attack_start AND m.attack_end 
                THEN 'Attack window'
            ELSE 'Benign baseline'
        END AS event_class,
        EXTRACT(HOUR FROM d.event_time) AS hour,
        COUNT(*) AS events
    FROM raw_device d
    LEFT JOIN dim_malicious_users m ON d.user = m.user_id
    GROUP BY event_class, hour
    ORDER BY event_class, hour
""").fetchdf().to_string())

=== USB activity volume by user class ===
  user_class  unique_users  device_events  avg_events_per_user
0  Malicious            70          44134                630.0
1     Benign           195         361246               1853.0

=== USB activity: in/out of attack windows ===
                              event_class  device_events
0                             Benign user         361246
1  Malicious user (outside attack window)          37047
2   Malicious user (during attack window)           7087

=== USB hour-of-day: attack-window events vs benign ===
        event_class  hour  events
0     Attack window     0      13
1     Attack window     1      22
2     Attack window     2      37
3     Attack window     3      50
4     Attack window     4      40
5     Attack window     5      33
6     Attack window     6      31
7     Attack window     7     144
8     Attack window     8     406
9     Attack window     9     658
10    Attack window    10     731
11    Attack window    11   

In [30]:
import pandas as pd

# Pivot the data to compare hour-by-hour
hour_compare = con.execute("""
    SELECT 
        EXTRACT(HOUR FROM d.event_time) AS hour,
        SUM(CASE 
            WHEN m.user_id IS NOT NULL 
              AND d.event_time BETWEEN m.attack_start AND m.attack_end 
            THEN 1 ELSE 0 END) AS attack_window_events,
        SUM(CASE 
            WHEN m.user_id IS NULL 
            THEN 1 ELSE 0 END) AS benign_events
    FROM raw_device d
    LEFT JOIN dim_malicious_users m ON d.user = m.user_id
    GROUP BY hour
    ORDER BY hour
""").fetchdf()

# Calculate hour-of-day percentages within each class
hour_compare['attack_pct'] = round(100 * hour_compare['attack_window_events'] / hour_compare['attack_window_events'].sum(), 2)
hour_compare['benign_pct'] = round(100 * hour_compare['benign_events'] / hour_compare['benign_events'].sum(), 2)
hour_compare['signal'] = round(hour_compare['attack_pct'] / hour_compare['benign_pct'].replace(0, 0.01), 2)

print("=== USB activity by hour: attack vs benign (% of class total) ===")
print(hour_compare[['hour', 'attack_window_events', 'attack_pct', 'benign_events', 'benign_pct', 'signal']].to_string(index=False))

print("\n📊 Signal column = attack_pct / benign_pct")
print("   > 1.0 means attack-window events are MORE concentrated at this hour")
print("   < 1.0 means LESS concentrated")

=== USB activity by hour: attack vs benign (% of class total) ===
 hour  attack_window_events  attack_pct  benign_events  benign_pct  signal
    0                  13.0        0.18          502.0        0.14    1.29
    1                  22.0        0.31          544.0        0.15    2.07
    2                  37.0        0.52          587.0        0.16    3.25
    3                  50.0        0.71          531.0        0.15    4.73
    4                  40.0        0.56          563.0        0.16    3.50
    5                  33.0        0.47          620.0        0.17    2.76
    6                  31.0        0.44          633.0        0.18    2.44
    7                 144.0        2.03         7133.0        1.97    1.03
    8                 406.0        5.73        28990.0        8.03    0.71
    9                 658.0        9.28        39597.0       10.96    0.85
   10                 731.0       10.31        36526.0       10.11    1.02
   11                 731.0       

In [31]:
print("Loading file.csv (193 MB) — DROPPING content field...")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_file AS
    SELECT 
        id, user, pc, filename,
        date AS date_text,
        TRY_STRPTIME(date, '%m/%d/%Y %H:%M:%S') AS event_time
    FROM read_csv_auto(
        '{(CERT_DIR / "file.csv").as_posix()}',
        sample_size=-1,
        ignore_errors=false
    )
""")
# Note: 'content' column intentionally not included in SELECT — it's dropped

cnt = con.execute("SELECT COUNT(*) FROM raw_file").fetchone()[0]
parsed = con.execute("SELECT COUNT(event_time) FROM raw_file").fetchone()[0]
print(f"✅ raw_file: {cnt:,} rows, {parsed:,} parsed ({cnt-parsed} failures)")

# Date range
print("\n=== Date range ===")
print(con.execute("SELECT MIN(event_time) AS earliest, MAX(event_time) AS latest FROM raw_file").fetchdf())

# File extension distribution (key insider threat signal)
print("\n=== Top file extensions accessed ===")
print(con.execute("""
    SELECT 
        LOWER(REGEXP_EXTRACT(filename, '\\.([a-zA-Z0-9]+)$', 1)) AS extension,
        COUNT(*) AS cnt
    FROM raw_file
    WHERE filename IS NOT NULL
    GROUP BY extension
    ORDER BY cnt DESC
    LIMIT 15
""").fetchdf())

# Malicious vs benign file activity
print("\n=== File access volume by user class ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN m.user_id IS NULL THEN 'Benign user'
            WHEN f.event_time BETWEEN m.attack_start AND m.attack_end 
                THEN 'Malicious user (during attack window)'
            ELSE 'Malicious user (outside attack window)'
        END AS event_class,
        COUNT(*) AS file_events
    FROM raw_file f
    LEFT JOIN dim_malicious_users m ON f.user = m.user_id
    GROUP BY event_class
""").fetchdf())

Loading file.csv (193 MB) — DROPPING content field...
✅ raw_file: 445,581 rows, 445,581 parsed (0 failures)

=== Date range ===
             earliest              latest
0 2010-01-02 07:23:14 2011-05-16 23:22:34

=== Top file extensions accessed ===
  extension     cnt
0       doc  285897
1       pdf   87953
2       txt   23033
3       jpg   22895
4       zip   22829
5       exe    2974

=== File access volume by user class ===
                              event_class  file_events
0   Malicious user (during attack window)         3647
1  Malicious user (outside attack window)        25036
2                             Benign user       416898


In [32]:
print("Loading email.csv (1.36 GB) — DROPPING content field, keeping metadata...")

# Drop existing if any
con.execute("DROP TABLE IF EXISTS raw_email")

# Load with content explicitly excluded
# size, attachments are useful as numeric features for insider threat ML
con.execute(f"""
    CREATE TABLE raw_email AS
    SELECT 
        id, user, pc,
        date AS date_text,
        TRY_STRPTIME(date, '%m/%d/%Y %H:%M:%S') AS event_time,
        "to" AS to_field,
        cc, bcc, "from" AS from_field,
        TRY_CAST(size AS BIGINT) AS size_bytes,
        TRY_CAST(attachments AS INTEGER) AS attachment_count
    FROM read_csv_auto(
        '{(CERT_DIR / "email.csv").as_posix()}',
        sample_size=-1,
        ignore_errors=false
    )
""")
# Note: 'content' column intentionally not selected — it's dropped

cnt = con.execute("SELECT COUNT(*) FROM raw_email").fetchone()[0]
parsed = con.execute("SELECT COUNT(event_time) FROM raw_email").fetchone()[0]
print(f"✅ raw_email: {cnt:,} rows, {parsed:,} parsed ({cnt-parsed} failures)")

# Storage saved by dropping content
print(f"\nSource file: 1.36 GB → in-memory table size:")
print(con.execute("""
    SELECT 
        round(estimated_size / 1e6, 1) AS table_mb
    FROM duckdb_tables() 
    WHERE table_name = 'raw_email'
""").fetchdf())

Loading email.csv (1.36 GB) — DROPPING content field, keeping metadata...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ raw_email: 2,629,979 rows, 2,629,979 parsed (0 failures)

Source file: 1.36 GB → in-memory table size:
   table_mb
0       2.6


In [33]:
# Date range and basic stats
print("=== Date range ===")
print(con.execute("SELECT MIN(event_time) AS earliest, MAX(event_time) AS latest FROM raw_email").fetchdf())

# Internal vs external email patterns (key insider threat signal)
# DTAA emails = internal; anything else = external
print("\n=== Email destination patterns ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN to_field IS NULL THEN 'No recipient'
            WHEN to_field LIKE '%@dtaa.com%' AND to_field NOT LIKE '%;%@%' THEN 'Internal only (single)'
            WHEN to_field LIKE '%@dtaa.com%' AND to_field LIKE '%;%' THEN 'Mixed internal/external'
            ELSE 'External only'
        END AS dest_pattern,
        COUNT(*) AS cnt
    FROM raw_email
    GROUP BY dest_pattern
    ORDER BY cnt DESC
""").fetchdf())

# Attachment distribution
print("\n=== Attachment count distribution ===")
print(con.execute("""
    SELECT 
        attachment_count,
        COUNT(*) AS emails
    FROM raw_email
    GROUP BY attachment_count
    ORDER BY attachment_count
    LIMIT 10
""").fetchdf())

# Email size distribution (large emails = potential exfiltration signal)
print("\n=== Email size buckets ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN size_bytes < 10000 THEN '< 10 KB'
            WHEN size_bytes < 100000 THEN '10-100 KB'
            WHEN size_bytes < 1000000 THEN '100 KB - 1 MB'
            WHEN size_bytes < 10000000 THEN '1-10 MB'
            ELSE '> 10 MB'
        END AS size_bucket,
        COUNT(*) AS emails
    FROM raw_email
    GROUP BY size_bucket
    ORDER BY MIN(size_bytes)
""").fetchdf())

=== Date range ===
             earliest              latest
0 2010-01-02 07:11:45 2011-05-16 21:16:26

=== Email destination patterns ===
              dest_pattern      cnt
0            External only  1044316
1   Internal only (single)   910707
2  Mixed internal/external   674956

=== Attachment count distribution ===
   attachment_count   emails
0                 0  2066683
1                 1   332335
2                 2   122620
3                 3    46157
4                 4    23450
5                 5    13316
6                 6     8800
7                 7     6515
8                 8     4309
9                 9     5794

=== Email size buckets ===
     size_bucket   emails
0        < 10 KB     1661
1      10-100 KB  2628171
2  100 KB - 1 MB      147


In [34]:
# Malicious email activity vs baseline
print("\n=== Email activity by user class ===")
print(con.execute("""
    SELECT 
        CASE 
            WHEN m.user_id IS NULL THEN 'Benign user'
            WHEN e.event_time BETWEEN m.attack_start AND m.attack_end 
                THEN 'Malicious user (during attack window)'
            ELSE 'Malicious user (outside attack window)'
        END AS event_class,
        COUNT(*) AS email_events,
        ROUND(AVG(attachment_count), 2) AS avg_attachments,
        ROUND(AVG(size_bytes / 1024.0), 0) AS avg_size_kb
    FROM raw_email e
    LEFT JOIN dim_malicious_users m ON e.user = m.user_id
    GROUP BY event_class
""").fetchdf())


=== Email activity by user class ===
                              event_class  email_events  avg_attachments  avg_size_kb
0   Malicious user (during attack window)          9858             0.34         29.0
1                             Benign user       2510146             0.41         29.0
2  Malicious user (outside attack window)        109975             0.35         29.0


In [35]:
# Scenario 3 users only (10 users) — do they show email volume anomalies?
print("=== Scenario 3 users: email volume during attack windows ===")
print(con.execute("""
    SELECT 
        e.user,
        COUNT(*) AS emails_during_attack,
        ROUND(AVG(e.attachment_count), 2) AS avg_attachments,
        MAX(e.attachment_count) AS max_attachments,
        ROUND(MAX(e.size_bytes) / 1024.0, 0) AS max_size_kb
    FROM raw_email e
    JOIN dim_malicious_users m ON e.user = m.user_id
    WHERE m.scenario = 3
      AND e.event_time BETWEEN m.attack_start AND m.attack_end
    GROUP BY e.user
    ORDER BY emails_during_attack DESC
""").fetchdf())

# Recipients (to + cc + bcc combined for Scenario 3 attacks — mass email signal)
print("\n=== Scenario 3 attacks: emails with many recipients ===")
print(con.execute("""
    SELECT 
        e.user,
        e.event_time,
        e.attachment_count,
        ROUND(e.size_bytes / 1024.0, 0) AS size_kb,
        LENGTH(e.to_field) - LENGTH(REPLACE(e.to_field, ';', '')) + 1 AS approx_recipient_count
    FROM raw_email e
    JOIN dim_malicious_users m ON e.user = m.user_id
    WHERE m.scenario = 3
      AND e.event_time BETWEEN m.attack_start AND m.attack_end
      AND LENGTH(e.to_field) - LENGTH(REPLACE(e.to_field, ';', '')) >= 5  -- 5+ recipients
    ORDER BY approx_recipient_count DESC
    LIMIT 15
""").fetchdf())

=== Scenario 3 users: email volume during attack windows ===
      user  emails_during_attack  avg_attachments  max_attachments  max_size_kb
0  JGT0221                    24             0.67                9         57.0
1  CSC0217                    24             0.00                0         40.0
2  MPM0220                    24             0.04                1         50.0
3  JLM0364                    23             0.17                3         58.0
4  MSO0222                    23             0.00                0         50.0
5  BSS0369                    21             0.29                6         64.0
6  JTM0223                    16             0.38                2         49.0
7  CCA0046                    15             0.40                3         65.0
8  BBS0039                    14             0.50                4         41.0
9  GTD0219                    13             0.31                2         62.0

=== Scenario 3 attacks: emails with many recipients ===
Em

In [36]:
import duckdb

# Build a unified view of all CERT activities — proves the activity-event pattern works
print("Building unified CERT activity view...")

con.execute("""
    CREATE OR REPLACE TABLE stg_cert_activities AS
    -- Logon events (Logon, Logoff)
    SELECT 
        id AS activity_id,
        event_time,
        user,
        pc,
        'authentication' AS activity_category,
        activity AS activity_subtype,
        NULL AS filename,
        NULL AS to_field,
        NULL AS from_field,
        NULL::INTEGER AS attachment_count,
        NULL::BIGINT AS size_bytes
    FROM raw_logon
    
    UNION ALL
    
    -- Device events (Connect, Disconnect)
    SELECT 
        id, event_time, user, pc,
        'device',
        activity,
        NULL, NULL, NULL, NULL, NULL
    FROM raw_device
    
    UNION ALL
    
    -- File access events
    SELECT 
        id, event_time, user, pc,
        'file',
        'File Access',
        filename,
        NULL, NULL, NULL, NULL
    FROM raw_file
    
    UNION ALL
    
    -- Email events
    SELECT 
        id, event_time, user, pc,
        'email',
        'Email Send',
        NULL,
        to_field, from_field, attachment_count, size_bytes
    FROM raw_email
""")

cnt = con.execute("SELECT COUNT(*) FROM stg_cert_activities").fetchone()[0]
print(f"✅ stg_cert_activities: {cnt:,} unified events")

# Validate unified row count = sum of inputs
expected = con.execute("""
    SELECT 
        (SELECT COUNT(*) FROM raw_logon) +
        (SELECT COUNT(*) FROM raw_device) +
        (SELECT COUNT(*) FROM raw_file) +
        (SELECT COUNT(*) FROM raw_email) AS expected_total
""").fetchone()[0]
print(f"   Expected: {expected:,} ({'✅ match' if cnt == expected else '❌ MISMATCH'})")

Building unified CERT activity view...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ stg_cert_activities: 4,335,799 unified events
   Expected: 4,335,799 (✅ match)


In [1]:
import duckdb

print("=== Identifier landscape: CICIDS vs CERT ===\n")

print("CICIDS2017 identifiers:")
# Open in READ-ONLY mode to avoid the lock conflict
cicids_db = duckdb.connect(
    r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_eda.duckdb",
    read_only=True
)
print(f"  Distinct Source IPs:      {cicids_db.execute('SELECT COUNT(DISTINCT \"Source IP\") FROM stg_cicids_final').fetchone()[0]:,}")
print(f"  Distinct Destination IPs: {cicids_db.execute('SELECT COUNT(DISTINCT \"Destination IP\") FROM stg_cicids_final').fetchone()[0]:,}")
print(f"  Date range:               2017-07-03 to 2017-07-07")
print(f"  Has user IDs?             NO — flow-based, no employee linkage")
cicids_db.close()

print("\nCERT r4.2 identifiers:")
print(f"  Distinct user_ids:        {con.execute('SELECT COUNT(DISTINCT user) FROM stg_cert_activities').fetchone()[0]:,}")
print(f"  Distinct PC names:        {con.execute('SELECT COUNT(DISTINCT pc) FROM stg_cert_activities').fetchone()[0]:,}")
print(f"  Date range:               2010-01-02 to 2011-05-17")
print(f"  Has IP addresses?         NO — uses PC names, no network info")

print("\n" + "="*60)
print("SHARED IDENTIFIERS BETWEEN CICIDS AND CERT")
print("="*60)
print("  None directly — datasets are temporally and identity-disjoint")
print()
print("UNIFICATION STRATEGY (the answer):")
print("  → Treat CICIDS and CERT as TWO TENANTS in the same warehouse")
print("  → Shared dimensions: dim_date, dim_time (calendar/clock)")
print("  → Source-specific dimensions: dim_user (CERT only), dim_asset (both)")
print("  → Source_system column on facts identifies origin")
print("  → Don't try to join events across datasets — it would be meaningless")

=== Identifier landscape: CICIDS vs CERT ===

CICIDS2017 identifiers:
  Distinct Source IPs:      17,005
  Distinct Destination IPs: 19,112
  Date range:               2017-07-03 to 2017-07-07
  Has user IDs?             NO — flow-based, no employee linkage

CERT r4.2 identifiers:


NameError: name 'con' is not defined

In [1]:
import duckdb
from pathlib import Path

# Reconnect to CERT database
INTERIM_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim")
CERT_DUCKDB_PATH = INTERIM_DIR / "cert_eda.duckdb"

con = duckdb.connect(str(CERT_DUCKDB_PATH))

# Verify our tables are still there
print("Tables in CERT DuckDB:")
print(con.execute("SHOW TABLES").fetchdf())

Tables in CERT DuckDB:
                  name
0  dim_malicious_users
1           raw_device
2            raw_email
3             raw_file
4             raw_ldap
5            raw_logon
6     raw_psychometric
7  stg_cert_activities


In [2]:
# Save each cleaned table to parquet
tables_to_export = [
    ('raw_psychometric', 'cert_psychometric.parquet'),
    ('raw_ldap', 'cert_ldap_history.parquet'),
    ('dim_malicious_users', 'cert_malicious_users.parquet'),
    ('stg_cert_activities', 'cert_activities_unified.parquet'),
]

for table_name, filename in tables_to_export:
    output_path = INTERIM_DIR / filename
    con.execute(f"""
        COPY (SELECT * FROM {table_name}) 
        TO '{output_path.as_posix()}' 
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    size_mb = output_path.stat().st_size / 1e6
    rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"✅ {filename}: {rows:,} rows, {size_mb:.1f} MB")

# Close cleanly
con.close()
print("\n✅ Connection closed")

✅ cert_psychometric.parquet: 1,000 rows, 0.0 MB
✅ cert_ldap_history.parquet: 16,743 rows, 0.0 MB
✅ cert_malicious_users.parquet: 70 rows, 0.0 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ cert_activities_unified.parquet: 4,335,799 rows, 124.8 MB

✅ Connection closed
